# Importação das bibliotecas necessárias

In [5]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [6]:
# Crie uma instância da classe
bucket = "bronze"
datamart = "compliance"
extrator_bronze = AlinharETL(bucket,datamart)

2024-10-17 11:16:38,610 - INFO - Iniciando Sessão Spark.


# Leitura das tabelas

In [7]:
df_compliance_estabelecimentos  = extrator_bronze.criar_view_temporaria('silver/dadosabertosreceita/flat_estabelecimentos','compliance_receita_estabelecimentos')

2024-10-17 11:16:39,414 - INFO - Aguarde... Criando View (silver/dadosabertosreceita/flat_estabelecimentos)
2024-10-17 11:16:45,208 - INFO - View criada com sucesso


In [8]:
df_compliance_empresas  = extrator_bronze.criar_view_temporaria('silver/dadosabertosreceita/dim_empresas','compliance_receita_empresas')

2024-10-17 11:16:45,214 - INFO - Aguarde... Criando View (silver/dadosabertosreceita/dim_empresas)
2024-10-17 11:16:45,606 - INFO - View criada com sucesso


In [9]:
df_compliance_motivos = extrator_bronze.criar_view_temporaria('silver/dadosabertosreceita/dim_motivos','compliance_receita_motivos')

2024-10-17 11:16:45,612 - INFO - Aguarde... Criando View (silver/dadosabertosreceita/dim_motivos)
2024-10-17 11:16:46,030 - INFO - View criada com sucesso


In [10]:
df_compliance_paises = extrator_bronze.criar_view_temporaria('silver/dadosabertosreceita/dim_paises','compliance_receita_paises')

2024-10-17 11:16:46,037 - INFO - Aguarde... Criando View (silver/dadosabertosreceita/dim_paises)
2024-10-17 11:16:46,368 - INFO - View criada com sucesso


In [11]:
df_compliance_cnaes = extrator_bronze.criar_view_temporaria('silver/dadosabertosreceita/dim_cnaes','compliance_receita_cnaes')

2024-10-17 11:16:46,373 - INFO - Aguarde... Criando View (silver/dadosabertosreceita/dim_cnaes)
2024-10-17 11:16:46,665 - INFO - View criada com sucesso


In [12]:
df_compliance_municipios = extrator_bronze.criar_view_temporaria('silver/dadosabertosreceita/dim_municipios','compliance_receita_municipios')

2024-10-17 11:16:46,671 - INFO - Aguarde... Criando View (silver/dadosabertosreceita/dim_municipios)
2024-10-17 11:16:46,940 - INFO - View criada com sucesso


# Tratamentos na tabela 

In [13]:
sql_query = """
SELECT 
    CONCAT(
        e.dsc_cnpj_basico,
        e.dsc_cnpj_ordem,
        e.dsc_cnpj_dv
    ) AS cnpj,
    e.dsc_situacao_cadastral,
    e.cd_cnae_fiscal_principal,
    e.cd_cnae_fiscal_secundaria,
    j.nm_razao_social,
    j.dsc_porte_empresa,
    c.descricao AS dsc_cnae_fiscal_principal,
    m.descricao AS dsc_municipio,
    e.nm_fantasia, 
    mot.descricao AS dsc_motivo_situacao_cadastral,
    e.dsc_telefone_1,
    e.dsc_telefone_2,
    e.dsc_matriz_filial
FROM compliance_receita_estabelecimentos e
LEFT JOIN compliance_receita_empresas j
    ON e.dsc_cnpj_basico = j.cd_cnpj_basico
LEFT JOIN compliance_receita_motivos mot
    ON e.cd_motivo_situacao_cadastral = mot.codigo
LEFT JOIN compliance_receita_paises p
    ON e.cd_pais = p.codigo
LEFT JOIN compliance_receita_cnaes c
    ON e.cd_cnae_fiscal_principal = c.codigo
LEFT JOIN compliance_receita_municipios m
    ON e.cd_municipio = m.codigo;
"""

df_join = extrator_bronze.carregar_dados_delta(sql_query)

2024-10-17 11:16:54,895 - INFO - Aguarde... Executando Query (Delta)
2024-10-17 11:16:54,897 - INFO - 
SELECT 
    CONCAT(
        e.dsc_cnpj_basico,
        e.dsc_cnpj_ordem,
        e.dsc_cnpj_dv
    ) AS cnpj,
    e.dsc_situacao_cadastral,
    e.cd_cnae_fiscal_principal,
    e.cd_cnae_fiscal_secundaria,
    j.nm_razao_social,
    j.dsc_porte_empresa,
    c.descricao AS dsc_cnae_fiscal_principal,
    m.descricao AS dsc_municipio,
    e.nm_fantasia, 
    mot.descricao AS dsc_motivo_situacao_cadastral,
    e.dsc_telefone_1,
    e.dsc_telefone_2,
    e.dsc_matriz_filial
FROM compliance_receita_estabelecimentos e
JOIN compliance_receita_empresas j
    ON e.dsc_cnpj_basico = j.cd_cnpj_basico
JOIN compliance_receita_motivos mot
    ON e.cd_motivo_situacao_cadastral = mot.codigo
JOIN compliance_receita_paises p
    ON e.cd_pais = p.codigo
JOIN compliance_receita_cnaes c
    ON e.cd_cnae_fiscal_principal = c.codigo
JOIN compliance_receita_municipios m
    ON e.cd_municipio = m.codigo;

2024-

# Gravação no datalake

In [10]:
extrator_bronze.caminho_tabela_delta = 's3a://gold/DadosAbertosReceita/FlatDadosAbertosReceitaFederal'

In [11]:
extrator_bronze.salvar_delta(df_join, 'overwrite')

2024-09-25 19:22:34,630 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 19:23:32,210 - INFO - Dados persistidos com sucesso
2024-09-25 19:23:32,210 - INFO - s3a://gold/DadosAbertosReceita/FlatDadosAbertosReceitaFederal
2024-09-25 19:23:32,211 - INFO - Aguarde... Realizando optimize
2024-09-25 19:23:39,190 - INFO - Optimize executado com sucesso.
2024-09-25 19:23:39,191 - INFO - Aguarde... Realizando vacuum
2024-09-25 19:23:54,123 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [12]:
extrator_bronze.parar_sessao()

2024-09-25 19:23:54,220 - INFO - Sessão Spark finalizada.
